# Per-sample QC and merging

In [ ]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import pandas as pd
import anndata as ad
#import scirpy as ir
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
#import awkward as ak
import vaeda
import hashlib
import numpy as np
import tensorflow as tf
from tqdm import tqdm

import gc

## Read in cellranger multi objects

This expects a directory structure similar to:

```
raw_data_dir/
    PATIENT_ID_1/
        outs/
            per_sample_outs/
                count/
                vdj_b/
                vdj_t/
    Patient_ID_2/

filtered_data_dir/
    PATIENT_ID_1/
```

This file is designed to be run on one sample at a time (for ease of adjusting quality control cutoffs); for multiple samples it is intended to be ran multiple times.

In [ ]:
sample_id = "SAMPLE_ID"

# Add your own data's naming quirks here to properly read in! 
sample_id_str = "" + sample_id + "_swapped"

per_sample_id_str = sample_id_str

# you need both
base_dir_rawdata = "/cellranger_raw_output_dir"
base_dir_filtered = "/dir_with_data_with_ambient_rna_removed"

sample_dir_rawdata = os.path.join(base_dir_rawdata, sample_id_str)
sample_dir = os.path.join(base_dir_filtered, sample_id)
print(sample_dir_rawdata)
print(per_sample_id_str)
print(sample_dir)

In [ ]:
expr_path = sample_dir
print(expr_path)
adata = sc.read_10x_mtx(expr_path, cache=False)
adata.var_names_make_unique()
print(adata.obs)
print(adata)
print(adata.X.shape)  # Should be (~6k or smthg it varies, n_genes)
print(adata.var.head()) # Gene names/features info

In [ ]:
print(adata.X.sum())

In [ ]:
barcode_obs1 = set(adata.obs.index)
print(f"Barcodes in adata.obs: {len(barcode_obs1)}")

In [ ]:
adata.obs.head()   # Cell barcodes & metadata
adata.obs_names 

### Light filtering and doublet detection

In [ ]:
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.filter_cells(adata, min_genes=200)

In [ ]:
sc.external.pp.scrublet(adata)

In [ ]:
adata = vaeda.vaeda(adata, seed=42)
adata.obs[['vaeda_scores', 'vaeda_calls']]

In [ ]:
non_singlets = adata.obs[(adata.obs['vaeda_calls'] != 'singlet') | (adata.obs['predicted_doublet'] != False)]

print(non_singlets.shape[0])
print(adata.obs[adata.obs['vaeda_calls'] == 'doublet'].shape[0])
print(adata.obs[adata.obs['predicted_doublet'] == True].shape[0])

In [ ]:
print(adata.obs.shape[0])
adata = adata[(adata.obs['vaeda_calls'] == 'singlet') & (adata.obs['predicted_doublet'] == False), :].copy()
print(adata.obs.shape[0])

### Annotate and control for % mitochondria, ribosomes, hemoglobin

In [ ]:
# Annotate mitochondrial genes
adata.var['mt'] = adata.var_names.str.startswith('MT-')

# Annotate ribosomal genes (human/human genes start with RPS or RPL)
adata.var['ribo'] = adata.var_names.str.startswith('RPS') | adata.var_names.str.startswith('RPL')

# Annotate hemoglobin genes
adata.var['hb'] = adata.var_names.isin(['HBA1','HBA2','HBB'])

In [ ]:
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt', 'ribo', 'hb'], inplace=True, log1p=True)

In [ ]:
#                0                       1                2                 3                 4
qc_metrics = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo', 'pct_counts_hb']

In [ ]:
mols_cutoff = adata.obs['total_counts'].quantile(0.95)
gene_cutoff = adata.obs['n_genes_by_counts'].quantile(0.95)
print(mols_cutoff)
print(gene_cutoff)

In [ ]:
#                        0                 1                2                 3                 4
#qc_metrics = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo', 'pct_counts_hb']
idx = 2
print(qc_metrics[idx])

field=qc_metrics[idx] #'pct_counts_mt' 'pct_counts_ribo'
#'total_counts'#'n_genes_by_counts'
adata_temp = adata[
    (adata.obs['total_counts'] > -1)

    & (adata.obs['n_genes_by_counts'] > 500) #const
    & (adata.obs['n_genes_by_counts'] < 4500) #const
    & (adata.obs['total_counts'] < 30000) #const
    & (adata.obs['total_counts'] > 1000) #const
]
print("Min:", adata_temp.obs[field].min())
print("Max:", adata_temp.obs[field].max())
print("Median:", np.median(adata_temp.obs[field]))
print("95th percentile:", np.percentile(adata_temp.obs[field], 95))
print("99th percentile:", np.percentile(adata_temp.obs[field], 99))

In [ ]:
#                0                       1                2                 3                 4
#qc_metrics = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo', 'pct_counts_hb']
idx = 2
counts = np.sort(adata.obs[qc_metrics[idx]])[::-1]
plt.plot(counts)
plt.yscale('log')
plt.xlabel("Barcodes")
plt.ylabel(qc_metrics[idx] + " (log)")
plt.title("Knee plot of " + qc_metrics[idx])

plt.axhline(y=13, color='red', linestyle='--', label='Example cutoff')

In [ ]:
#                0                       1                2                 3                 4
qc_metrics = ['n_genes_by_counts', 'total_counts', 'pct_counts_mt', 'pct_counts_ribo', 'pct_counts_hb']
idx = 2

metric = qc_metrics[idx]
data = adata_temp.obs[metric]

plt.hist(data, bins=100, color='skyblue')
#plt.axvline(x=30, color='red', linestyle='--', label='Example cutoff')
plt.xlabel(metric)
plt.ylabel('Number of cells')
#plt.legend()
plt.show()

In [ ]:
fig_scatter, axs_scatter = plt.subplots(nrows=1, ncols=2, figsize=(12, 5))
sc.pl.scatter(adata, ax=axs_scatter[0], x='total_counts', y='pct_counts_mt',show=False, title='% MT as function of total counts')
sc.pl.scatter(adata, ax=axs_scatter[1],  x='total_counts', y='n_genes_by_counts',show=False, title='Genes as function of total counts')
plt.tight_layout()
plt.show()

In [ ]:
#genes (n_genes) and molecules (total_counts): hold standard across samples

print("Before filtering:", adata.obs.shape[0])

adata_graph = adata[
    (adata.obs['total_counts'] > -1)
    
    & (adata.obs['n_genes_by_counts'] > 500) #const
    & (adata.obs['n_genes_by_counts'] < 4500) #const
    & (adata.obs['total_counts'] < 30000) #const
    & (adata.obs['total_counts'] > 1000) #const
    & (adata.obs['pct_counts_mt'] < 20)
    & (adata.obs['pct_counts_ribo'] < 35)
    & (adata.obs['pct_counts_hb'] < 0.01)
].copy()

fig_violin, axs_violin = plt.subplots(1, len(qc_metrics), figsize=(5 * len(qc_metrics), 5))

for i, metric in enumerate(qc_metrics):
    sns.violinplot(
        y=adata_graph.obs[metric], inner=None, ax=axs_violin[i], color='skyblue'
    )

    sns.stripplot(
        y=adata_graph.obs[metric], 
        ax=axs_violin[i], 
        size=3, 
        jitter=0.3,
        alpha=0.7,
        hue=adata_graph.obs['vaeda_scores'],  # or any other column
        palette='viridis',
        dodge=False
        #legend=True  # to avoid automatic legend in each subplot
    )
    axs_violin[i].set_title(metric)
    axs_violin[i].set_ylabel(metric)
    axs_violin[i].set_xlabel('')

print(adata.obs.shape[0])
print("After filtering:", adata_graph.obs.shape[0])

In [ ]:
print(adata.obs.columns)

In [ ]:
print(adata.X.shape)
print(np.min(adata.X), np.max(adata.X), np.mean(adata.X))

In [ ]:
adata=adata_graph.copy()
print(len(adata.obs))
print(adata.X.shape)
print(np.min(adata.X), np.max(adata.X), np.mean(adata.X))

## QC of VDJ data

### Read in VDJ

In [ ]:
print(sample_dir)
print(sample_dir_rawdata)
print(per_sample_id_str)

In [ ]:
vdj_b_path = os.path.join(sample_dir_rawdata, "outs", "per_sample_outs", per_sample_id_str, "vdj_b", "filtered_contig_annotations.csv")
vdj_t_path = os.path.join(sample_dir_rawdata, "outs", "per_sample_outs", per_sample_id_str, "vdj_t", "filtered_contig_annotations.csv")

vdj_b_loaded = pd.read_csv(vdj_b_path)
vdj_t_loaded = pd.read_csv(vdj_t_path)

print(vdj_t_path)
print(vdj_b_loaded.columns.tolist())

In [ ]:
print(vdj_t_loaded.shape[0])
print(vdj_b_loaded.shape[0])

In [ ]:
vdj_t_loaded

In [ ]:
t_chains = ['TRA', 'TRB']
b_chains = ['IGH', 'IGK', 'IGL']

### Filter out low reads and UMIs

In [ ]:
# histogram to visualize UMI distr
sns.histplot(vdj_t_loaded['umis'], bins=50, log_scale=(False, True),discrete=True)
plt.xlim((-0, 50))
plt.axvline(x=2, color='red', linestyle='--', label='UMI = 1')
plt.legend()
plt.xlabel('UMI count')
plt.ylabel('Number of contigs')
plt.title('UMI distribution in TCR data')
plt.show()

In [ ]:
sns.scatterplot(data=vdj_b_loaded, x='umis', y='reads', alpha=0.5)
plt.xlim((-0, 10))
#plt.ylim((-0, 100))
plt.axvline(x=1, color='red', linestyle='--')
plt.axhline(y=20, color='red', linestyle='--')

In [ ]:
# Filter productive only (important to keep only productive TCRs/BCRs)
vdj_b_filtered = vdj_b_loaded[
    (vdj_b_loaded['high_confidence'] == True) &
    (vdj_b_loaded['reads'] > 10) &
    (vdj_b_loaded['umis'] > 1) &
    (vdj_b_loaded['v_gene'].notna()) &
    (vdj_b_loaded['j_gene'].notna()) &
    (vdj_b_loaded['chain'].isin(b_chains))
].copy()

vdj_t_filtered = vdj_t_loaded[
    (vdj_t_loaded['high_confidence'] == True) &
    (vdj_t_loaded['reads'] > 10) &
    (vdj_t_loaded['umis'] > 1) &
    (vdj_t_loaded['v_gene'].notna()) &
    (vdj_t_loaded['j_gene'].notna()) &
    (vdj_t_loaded['chain'].isin(t_chains))
].copy()

print(vdj_t_filtered.shape[0])
print(vdj_b_filtered.shape[0])

In [ ]:
vdj_b_suffixed = vdj_b_filtered.add_suffix('_b')
vdj_t_suffixed = vdj_t_filtered.add_suffix('_t')
vdj_t_suffixed = vdj_t_suffixed.rename(columns={'barcode_t': 'barcode'})
vdj_b_suffixed = vdj_b_suffixed.rename(columns={'barcode_b': 'barcode'})

if vdj_t_suffixed.shape[0] == 0:
    print("Warning: nothing passed filter for vdj_t.")

if vdj_b_suffixed.shape[0] == 0:
    print("Warning: nothing passed filter for vdj_b.")

print("vdj_t length: " + str(vdj_t_suffixed.shape[0]))
print("vdj_b length: " + str(vdj_b_suffixed.shape[0]))
print(vdj_t_suffixed.columns)
print(vdj_b_suffixed.columns)

### Hasher functions to assign clonotypes based on V/J gene usage and cellranger output

In [ ]:
# will result in different clonotype assignments than cellranger, merges some and splits on others that cellranger doesn't, use with discretion

def stable_hash(*args):
    args = ['' if (pd.isna(arg)) else str(arg) for arg in args]
    s = "_".join(args)
    return hashlib.md5(s.encode('utf-8')).hexdigest()

def all_fields_NA(cols_to_hash):
    return all((c is pd.NA or c == '' or pd.isna(c)) for c in cols_to_hash)

### Aggregate data based on barcode, pick the most confident entry for each unique barcode

In [ ]:
def aggregate_chain_info(df, chain_types, bt, aggregate_fields=None, concat_fields = None, clonotype_id_hash_fields=None):
    """
    Aggregates VDJ information for specified chains (e.g., TRA, TRB).
    
    Arguments:
        df: DataFrame filtered to productive contigs.
        chain_types: list of chain types to aggregate (e.g., ['TRA', 'TRB']).
        aggregate_fields: list of fields to aggregate per chain; not including 'raw_clonotype_id' which is aggregated automatically.
                        If None, defaults to ['cdr3', 'v_gene', 'j_gene'].
        concat_fields: list of fields to store all contigs for (by concatenation).
                        If None, no concatenation is performed.
    
    Returns:
        pd.Series with aggregated info for each barcode.
    """

    if aggregate_fields is None:
        aggregate_fields = ['cdr3', 'v_gene', 'j_gene']
    
    if clonotype_id_hash_fields is None:
        clonotype_id_hash_fields = ['cdr3', 'v_gene', 'j_gene']

    result = {}
    hash_fields = []
    reads_col = f'umis_{bt}'
    
    if f'raw_clonotype_id_{bt}' in df.columns:
        clone_subset = df[df[f'raw_clonotype_id_{bt}'].notna()]

        if not clone_subset.empty:
            if reads_col in clone_subset.columns and not clone_subset[reads_col].isna().all():
                top_idx = clone_subset[reads_col].idxmax()
            else:
                top_idx = clone_subset.index[0]

            result[f'comb_raw_clonotype_id_{bt}'] = clone_subset.loc[top_idx, f'raw_clonotype_id_{bt}']
        else:
            result[f'comb_raw_clonotype_id_{bt}'] = pd.NA

        if (concat_fields is not None) and ('raw_clonotype_id' in concat_fields):
            ids = df[f'raw_clonotype_id_{bt}'].dropna().unique()
            result[f'comb_raw_clonotype_id_concat_all_{bt}'] = ';'.join(ids) if len(ids) else pd.NA

    for chain in chain_types:
        subset = df[df[f'chain_{bt}'] == chain]

        for rawfield in aggregate_fields:
            field = (rawfield + f"_{bt}")
            col_name = f"{field.lower()}_{chain.lower()}"
            concat_col_name = col_name + "_concat_all"
            if subset.empty or field not in subset.columns:
                result[col_name] = pd.NA

            # choose the most confident entry (e.g., highest UMI) or aggregate all
            else:
                if reads_col in subset.columns and not subset[reads_col].isna().all():
                    top_idx = subset[reads_col].idxmax()
                else:
                    top_idx = subset.index[0]
                result[col_name] = subset.loc[top_idx, field]
                
            # concatenate if field is present
            if (concat_fields is not None) and (field in subset.columns) and (rawfield in concat_fields):
                values = subset[field].dropna().unique()
                if len(values) == 0:
                    result[concat_col_name] = pd.NA
                else:
                    result[concat_col_name] = ';'.join(str(v) for v in values)

        for c in clonotype_id_hash_fields:
            hash_fields.append(c + f'_{bt}_{chain.lower()}')

    # (optional) include a representative raw_clonotype_id if available
    
    cols_to_hash = [result.get(key, '') for key in hash_fields]
    if all_fields_NA(cols_to_hash):
        result[f'my-hash_clonotype_id_{bt}'] = pd.NA
    else:
        result[f'my-hash_clonotype_id_{bt}'] = f"clone_{stable_hash(*cols_to_hash)}"
        
    return pd.Series(result)

In [ ]:
# example fields that we want to aggregate at barcode-level
aggarr = ['cdr3', 'cdr3_nt', 'v_gene', 'd_gene', 'j_gene', 'c_gene', 'exact_subclonotype_id',
          'raw_consensus_id', 'full_length', 'raw_clonotype_id']
concatarr = ['cdr3', 'cdr3_nt', 'v_gene', 'j_gene', 'raw_clonotype_id']
chain_arr_tcr = t_chains
chain_arr_bcr = b_chains

vdj_t_agg = (
    vdj_t_suffixed
    .groupby('barcode')
    .apply(aggregate_chain_info, chain_arr_tcr, "t", aggarr, concatarr)
    .reset_index()
)

vdj_b_agg = (
    vdj_b_suffixed
    .groupby('barcode')
    .apply(aggregate_chain_info, chain_arr_bcr, "b", aggarr, concatarr)
    .reset_index()
)

print(vdj_t_suffixed.columns)
print(vdj_t_agg.columns)

In [ ]:
vdj_temp = vdj_t_agg[['comb_raw_clonotype_id_t', 'my-hash_clonotype_id_t']]
vdj_temp

In [ ]:
print(vdj_t_agg.columns)
assert vdj_t_agg.index.name is None

### Join VDJ(b) and VDJ(t) on barcodes

In [ ]:
vdj_b = vdj_b_agg.set_index('barcode', drop=False)
vdj_t = vdj_t_agg.set_index('barcode', drop=False)
print(vdj_b.columns)
print(vdj_t.columns)

In [ ]:
vdj_combined = vdj_t.join(vdj_b, how='outer', lsuffix='_t', rsuffix='_b')
(vdj_combined[['barcode_b', 'comb_raw_clonotype_id_b', 'barcode_t', 'comb_raw_clonotype_id_t']])

In [ ]:
assert vdj_combined.index.name == 'barcode', "Index failed the name check."
assert len(vdj_combined) == vdj_combined.index.nunique(), "Some rows failed the length check."

# At least one barcode is present
at_least_one = vdj_combined['barcode_b'].notna() | vdj_combined['barcode_t'].notna()

# barcode_b must equal index if it exists
barcode_b_ok = np.where(
    vdj_combined['barcode_b'].notna(),
    vdj_combined.index == vdj_combined['barcode_b'],
    True
)

# barcode_t must equal index if it exists
barcode_t_ok = np.where(
    vdj_combined['barcode_t'].notna(),
    vdj_combined.index == vdj_combined['barcode_t'],
    True
)

# Combine all checks
all_checks = at_least_one & barcode_b_ok & barcode_t_ok

# Assert all rows pass
assert all_checks.all(), "Some rows failed the barcode check."

In [ ]:
print(adata.obs.index[:5])
print(vdj_combined.index[:5]) 

In [ ]:
barcode_obs = set(adata.obs.index)
barcode_vdj = set(vdj_combined.index)
print(f"Barcodes in adata.obs: {len(barcode_obs)}")
print(f"Barcodes in vdj_combined: {len(barcode_vdj)}")
print(f"Intersection: {len(barcode_obs.intersection(barcode_vdj))}")

In [ ]:
list(barcode_obs)[:10]
list(barcode_vdj)[:10]

In [ ]:
len(set(x.split('-')[0] for x in barcode_obs) & set(x.split('-')[0] for x in barcode_vdj))

### Merge VDJ and RNA data on barcodes

In [ ]:
assert len(adata.obs) == adata.obs.index.nunique()
adata_backup = adata.copy()

In [ ]:
# Sanity: ensure barcodes are strings
vdj_combined.index = vdj_combined.index.astype(str)
adata.obs.index = adata.obs.index.astype(str)

adata.obs = adata.obs.join(vdj_combined, how="left")
adata.obs.head()

In [ ]:
adata.obs.columns

## Write merged adata out to .h5ad object

In [ ]:
na_string = '<NA>'

# Drop completely empty columns
adata.obs = adata.obs.dropna(axis=1, how='all')

# Convert all nullable dtypes to safe types for h5ad
for col in adata.obs.columns:
    dtype = str(adata.obs[col].dtype)
    
    if dtype == "Int64":  # nullable ints
        adata.obs[col] = adata.obs[col].to_numpy(dtype=float)
    elif dtype == "Float64":  # nullable floats
        adata.obs[col] = adata.obs[col].to_numpy(dtype=float)
    elif dtype == "boolean":  # nullable bools
        # Convert NaNs to False (adata cannot store NA bool)
        adata.obs[col] = adata.obs[col].to_numpy(dtype=bool)
    elif dtype == "string":  # nullable strings
        # Optionally replace <NA> with a string
        adata.obs[col] = adata.obs[col].fillna(na_string).astype(str)

# Check for remaining problem columns
problem_cols = [
    (col, str(adata.obs[col].dtype))
    for col in adata.obs.columns
    if str(adata.obs[col].dtype) in ["Int64", "Float64", "boolean", "string"]
]

if problem_cols:
    print("Problematic columns still present:")
    for col, dtype in problem_cols:
        print(f"{col:30} {dtype}")
else:
    print("All obs columns safe for h5ad!")

In [ ]:
adata.obs.head()

In [ ]:
sc.settings.allow_write_nullable_strings = True

for col in adata.obs.columns:
    if str(adata.obs[col].dtype) == "string" or adata.obs[col].dtype == object:
        adata.obs[col] = adata.obs[col].apply(lambda x: str(x) if pd.notna(x) else "<NA>")

# Save to .h5ad object
save_dir = '/your_save_dir'
outfile = f"{save_dir}/{sample_id}.h5ad"
print("Saving to:", outfile)
adata.write(outfile)

### Save the violin and scatter plots from earlier (optinal)

In [ ]:
fig_save_dir = "/your_figures_dir"
print(fig_save_dir)
os.makedirs(fig_save_dir, exist_ok=True)

In [ ]:
fig_scatter.savefig(os.path.join(fig_save_dir, "scatter.png"))
fig_violin.savefig(os.path.join(fig_save_dir, "violin.png"))

## Test to ensure saving worked

In [ ]:
adata.obs.head()

In [ ]:
adata_test = sc.read_h5ad(outfile)
adata_test.obs.head()

In [ ]:
adata.obs.columns